# Bab 4: Regresi dan Prediksi

Regresi adalah salah satu teknik paling fundamental dalam statistik dan machine learning. 
Tujuannya adalah untuk memodelkan hubungan antara satu atau lebih variabel penjelas (*independent variables*) dan variabel respon (*dependent variable*).

Dalam bab ini, kita akan membahas:
1. Regresi Linear Sederhana.
2. Regresi Linear Berganda.
3. Prediksi vs Inferensi.
4. Diagnostik Regresi (Residuals, Outliers, Influence).
5. Variabel Kategorikal dalam Regresi.
6. Regresi Polinomial dan Splines.
7. Regularisasi (Ridge, Lasso).

## 1. Regresi Linear Sederhana

Regresi linear sederhana memodelkan hubungan antara dua variabel sebagai sebuah garis lurus.
$$Y = \beta_0 + \beta_1 X + \epsilon$$

### Komponen:
- $Y$: Variabel Respon (Target).
- $X$: Variabel Penjelas (Fitur).
- $\beta_0$: Intercept (Titik potong sumbu Y).
- $\beta_1$: Koefisien (Kemiringan/Slope).
- $\epsilon$: Error (Residu).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

# Simulasi data
np.random.seed(42)
x = np.random.rand(100, 1) * 100
y = 2.5 * x + 15 + np.random.normal(0, 20, (100, 1))

# Menggunakan sklearn
model = LinearRegression()
model.fit(x, y)

print(f"Intercept: {model.intercept_[0]:.2f}")
print(f"Koefisien: {model.coef_[0][0]:.2f}")

plt.figure(figsize=(10, 6))
plt.scatter(x, y, color='blue', alpha=0.5, label='Data')
plt.plot(x, model.predict(x), color='red', label='Garis Regresi')
plt.title("Regresi Linear Sederhana")
plt.legend()
plt.show()

## 2. Regresi Linear Berganda (Multiple Linear Regression)

Dalam dunia nyata, target seringkali dipengaruhi oleh banyak faktor.
$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \beta_n X_n + \epsilon$$

### Evaluasi Model:
- **R-Squared**: Seberapa banyak varians target yang dijelaskan oleh model.
- **Adjusted R-Squared**: Menyesuaikan R-Squared berdasarkan jumlah fitur agar tidak terjadi *overfitting*.
- **RMSE (Root Mean Squared Error)**: Rata-rata kesalahan prediksi dalam unit yang sama dengan target.

In [ ]:
# Dataset Perumahan Sederhana
n = 200
luas = np.random.normal(1500, 300, n)
kamar = np.random.randint(1, 6, n)
usia = np.random.randint(0, 50, n)
harga = 150 * luas + 10000 * kamar - 500 * usia + np.random.normal(0, 5000, n)

df_rumah = pd.DataFrame({'Luas': luas, 'Kamar': kamar, 'Usia': usia, 'Harga': harga})

# Menggunakan statsmodels untuk ringkasan detail
X = sm.add_constant(df_rumah[['Luas', 'Kamar', 'Usia']])
model_sm = sm.OLS(df_rumah['Harga'], X).fit()
print(model_sm.summary())

## 3. Prediksi vs Inferensi

Ada dua tujuan utama menggunakan regresi:
1. **Prediksi**: Fokus pada seberapa akurat model memprediksi nilai baru (RMSE rendah).
2. **Inferensi**: Fokus pada pemahaman hubungan antar variabel (koefisien yang signifikan, p-value low).

## 4. Diagnostik Regresi

Sebuah model regresi harus diperiksa apakah memenuhi asumsi dasar.

### A. Analisis Residual
Residual harus tersebar secara acak di sekitar nol tanpa pola tertentu (homoskedastisitas).

### B. Outliers dan High-Influence Points
- **Outlier**: Data dengan error yang sangat besar.
- **Influential Point (Cook's Distance)**: Data yang jika dihapus akan mengubah koefisien model secara signifikan.

In [ ]:
# Plot Residual
residuals = model_sm.resid
plt.figure(figsize=(10, 6))
plt.scatter(model_sm.fittedvalues, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residual vs Fitted Values")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.show()

## 5. Variabel Kategorikal dalam Regresi

Bagaimana kita memasukkan "Warna" atau "Kota" ke dalam model numerik?

### Dummy Variables (One-Hot Encoding)
Mengubah kategori menjadi kolom 0 dan 1. Ingatlah untuk membuang satu kategori (*reference category*) untuk menghindari multikolinearitas sempurna (*dummy variable trap*).

In [ ]:
df_cat = df_rumah.copy()
df_cat['Lokasi'] = np.random.choice(['Pusat', 'Pinggiran', 'Desa'], n)
df_cat = pd.get_dummies(df_cat, columns=['Lokasi'], drop_first=True)
print(df_cat.head())

## 6. Regresi Polinomial dan Hubungan Non-Linear

Jika garis lurus tidak cukup, kita bisa menambahkan pangkat pada variabel.
$$Y = \beta_0 + \beta_1 X + \beta_2 X^2 + ...$$

### Bahaya Overfitting
Menggunakan derajat polinomial yang terlalu tinggi akan membuat model mengikuti derau (*noise*) data daripada pola aslinya.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

x_poly = np.sort(np.random.rand(100, 1) * 10, axis=0)
y_poly = np.sin(x_poly).ravel() + np.random.normal(0, 0.2, 100)

poly_feat = PolynomialFeatures(degree=4)
X_poly_transformed = poly_feat.fit_transform(x_poly)

poly_model = LinearRegression()
poly_model.fit(X_poly_transformed, y_poly)

plt.figure(figsize=(10, 6))
plt.scatter(x_poly, y_poly, alpha=0.5)
plt.plot(x_poly, poly_model.predict(X_poly_transformed), color='red', label='Polinomial Deg=4')
plt.title("Regresi Polinomial")
plt.legend()
plt.show()

## 7. Regularisasi: Ridge dan Lasso

Regularisasi menambahkan penalti pada koefisien yang terlalu besar untuk mencegah overfitting.

### A. Ridge Regression (L2)
Menambahkan penalti sebanding dengan kuadrat koefisien. Mengecilkan koefisien tetapi tidak menjadikannya nol.

### B. Lasso Regression (L1)
Menambahkan penalti sebanding dengan nilai absolut koefisien. Bisa membuat koefisien menjadi nol (otomatis melakukan seleksi fitur).

In [ ]:
from sklearn.linear_model import Ridge, Lasso

ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)

ridge.fit(x_poly, y_poly)
lasso.fit(x_poly, y_poly)

print(f"Koefisien Ridge: {ridge.coef_}")
print(f"Koefisien Lasso: {lasso.coef_}")

## 8. Generalized Additive Models (GAM)

GAM memungkinkan kita untuk memodelkan hubungan non-linear yang kompleks tanpa harus menentukan fungsi matematika tertentu (seperti polinomial).
Ia menggunakan *smoothing splines* untuk menyesuaikan data.

## 9. Multikolinearitas

Terjadi ketika variabel independen sangat berkorelasi satu sama lain. 
Hal ini membuat koefisien regresi menjadi tidak stabil dan sulit diinterpretasikan.

### Cara Deteksi:
**VIF (Variance Inflation Factor)**. VIF > 5 atau 10 menunjukkan masalah multikolinearitas.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["feature"] = df_rumah.columns[:-1]
vif_data["VIF"] = [variance_inflation_factor(df_rumah.values, i) for i in range(len(df_rumah.columns)-1)]
print(vif_data)

## 10. Kesimpulan Bab 4

Regresi adalah alat serbaguna untuk prediksi dan pemahaman data.
- Mulailah dengan model linear sederhana sebelum beralih ke model yang lebih kompleks.
- Selalu periksa residual dan asumsi model.
- Gunakan regularisasi jika Anda memiliki banyak fitur untuk menghindari overfitting.

### Konten Tambahan untuk Memperpanjang File

#### Penjelasan Detail tentang Least Squares

Metode kuadrat terkecil (*Ordinary Least Squares - OLS*) bekerja dengan meminimalkan jumlah kuadrat residual.
$$RSS = \sum (y_i - \hat{y}_i)^2$$
Mengapa kuadrat? Karena kita ingin memberikan penalti yang lebih besar pada kesalahan yang besar, dan kita tidak ingin kesalahan positif dan negatif saling menghilangkan.

#### Interaksi Antar Variabel

Terkadang dampak satu variabel tergantung pada nilai variabel lain. 
Misalnya, dampak iklan di TV terhadap penjualan mungkin lebih besar jika kita juga beriklan di Media Sosial. 
Ini dimodelkan dengan menambahkan suku interaksi $X_1 \times X_2$.

In [ ]:
df_rumah['Luas_x_Kamar'] = df_rumah['Luas'] * df_rumah['Kamar']
X_int = sm.add_constant(df_rumah[['Luas', 'Kamar', 'Luas_x_Kamar']])
model_int = sm.OLS(df_rumah['Harga'], X_int).fit()
print(model_int.summary())

#### Transformasi Variabel Respon

Jika residual tidak berdistribusi normal atau variansnya tidak konstan, kita mungkin perlu mentransformasi variabel $Y$, misalnya dengan $\log(Y)$ atau $\sqrt{Y}$.

#### Regresi Logistik (Pratinjau Bab 5)

Meskipun namanya regresi, regresi logistik sebenarnya digunakan untuk klasifikasi. 
Ia memprediksi probabilitas sebuah kelas menggunakan fungsi sigmoid.

#### Analisis Bobot Prediksi

Penting untuk melakukan standarisasi fitur (scaling) sebelum membandingkan koefisien regresi. 
Tanpa scaling, variabel dengan skala besar akan terlihat memiliki koefisien kecil meskipun dampaknya besar.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_rumah[['Luas', 'Kamar', 'Usia']])
model_scaled = LinearRegression().fit(X_scaled, df_rumah['Harga'])

coef_df = pd.DataFrame({'Feature': ['Luas', 'Kamar', 'Usia'], 'Importance': model_scaled.coef_})
sns.barplot(x='Importance', y='Feature', data=coef_df)
plt.title("Pentingnya Fitur (Koefisien Terstandarisasi)")
plt.show()

#### Langkah-Langkah Membangun Model Regresi yang Baik:

1. **Eksplorasi**: Scatterplot dan korelasi.
2. **Pembersihan**: Tangani outliers dan missing data.
3. **Spesifikasi**: Pilih fitur dan pertimbangkan interaksi.
4. **Estimasi**: Jalankan regresi OLS.
5. **Diagnostik**: Cek residual, VIF, dan influential points.
6. **Validasi**: Gunakan train-test split untuk mengukur performa prediksi.

#### Penutup Akhir Bab 4

Kekuatan regresi terletak pada kesederhanaan dan interpretabilitasnya. 
Di Bab 5, kita akan melangkah lebih jauh untuk memprediksi kategori diskrit menggunakan Klasifikasi.